# Austrian law RAG – 02: compare all retrievers (milestone 2)

**Before you start: Runtime → Change runtime type → T4 GPU.** Then *Runtime → Run all*.
Takes about 10 minutes (mostly downloading three models).

Compares, on your questions:

| Family | Retrievers |
| --- | --- |
| Lexical | BM25 on words · on German stems · on 4-grams of stems (handles compounds like *Ruhepause*) |
| Dense | multilingual E5 · BGE-M3 |
| Hybrid | the first-run mix (BM25 + E5) · weighted mixes of BM25 4-grams with each dense model |
| Reranking | a cross-encoder (BGE reranker v2 m3) re-scores each dense model's top 30 chunks |

Results go to `MyDrive/austrian-law-rag/results_m2/`.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE – switch the runtime to T4 GPU first')

In [ ]:
import os
if not os.path.exists('/content/austrian-law-rag'):
    !git clone -q https://github.com/zarekasif/austrian-law-rag.git /content/austrian-law-rag
else:
    !git -C /content/austrian-law-rag pull -q
%cd /content/austrian-law-rag
!pip install -q -e ".[dense]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/austrian-law-rag'
CORPUS = f'{DATA}/corpus.jsonl'
QUESTIONS = f'{DATA}/questions.jsonl' if os.path.exists(f'{DATA}/questions.jsonl') else 'data/questions.jsonl'
OUT = f'{DATA}/results_m2'
print('questions from:', QUESTIONS)

## Run all retrievers

In [ ]:
!python -m lawrag.experiments --corpus "$CORPUS" --questions "$QUESTIONS" --out "$OUT" --cache "$DATA/embeddings"

## Which questions does each retriever get wrong?

Each cell is the rank of the correct § (1 = top result); empty = not found in the top 100.

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 250)
df = pd.read_csv(f'{OUT}/per_question.csv')
df

Copy the table printed by the "Run all retrievers" cell into the chat. If you want, also download
`per_question.csv` from `MyDrive/austrian-law-rag/results_m2/` and upload it.